# 🚀 AdaptiveSLM Training on Kaggle

**Research-Grade Small Language Model**

Novel Contributions:
1. **PAKD** - Profile-Aware Knowledge Distillation
2. **ACC** - Adaptive Context Compression
3. **SCPD** - Semantic Cache with Priority Decay

---

**Hardware Options (choose one):**
- GPU T4 x2 (30GB VRAM total) ✓
- GPU P100 (16GB VRAM) ✓
- TPU v5e-8 (fastest!) ✓

In [ ]:
# Check hardware
import torch
import os

print(f"PyTorch: {torch.__version__}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"GPU Count: {torch.cuda.device_count()}")
    DEVICE = "cuda"
    USE_TPU = False
else:
    try:
        import torch_xla.core.xla_model as xm
        DEVICE = xm.xla_device()
        USE_TPU = True
        print(f"TPU: {DEVICE}")
    except:
        DEVICE = "cpu"
        USE_TPU = False
        print("Warning: No GPU/TPU found, using CPU")

In [ ]:
# Install dependencies
!pip install -q transformers>=4.36.0 datasets>=2.16.0 accelerate>=0.25.0
!pip install -q bitsandbytes>=0.41.0 wandb>=0.16.0 sentencepiece
!pip install -q huggingface_hub

## 1. Configuration

In [ ]:
from dataclasses import dataclass, field
from typing import Optional, List, Dict

@dataclass
class TrainingConfig:
    """Configuration optimized for Kaggle GPUs"""
    
    # Model Architecture (MobileLLM-style: deep and thin)
    vocab_size: int = 32000
    hidden_size: int = 576          # Thin
    num_layers: int = 30            # Deep (vs typical 12)
    num_attention_heads: int = 9
    num_key_value_heads: int = 3    # GQA
    intermediate_size: int = 1536
    max_position_embeddings: int = 2048
    
    # Training (optimized for T4x2 / P100)
    batch_size: int = 8             # Per GPU
    gradient_accumulation_steps: int = 16  # Effective batch = 128
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    warmup_steps: int = 500
    max_steps: int = 10000          # ~8 hours on T4
    
    # Knowledge Distillation
    teacher_model: str = "Qwen/Qwen2.5-7B-Instruct"
    distillation_alpha: float = 0.5
    temperature: float = 2.0
    
    # PAKD (our novel contribution)
    pakd_enabled: bool = True
    
    # Memory Optimization
    use_gradient_checkpointing: bool = True
    use_mixed_precision: bool = True  # bfloat16 on T4/P100
    
    # Paths
    output_dir: str = "/kaggle/working/checkpoints"
    
config = TrainingConfig()
print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps}")

## 2. Model Architecture

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps
    
    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        x = x * torch.rsqrt(variance + self.eps)
        return self.weight * x

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_position_embeddings=2048, base=10000):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.max_seq_len_cached = max_position_embeddings
        t = torch.arange(self.max_seq_len_cached).float()
        freqs = torch.einsum("i,j->ij", t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())
    
    def forward(self, x, seq_len):
        return self.cos_cached[:seq_len], self.sin_cached[:seq_len]

def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = self.hidden_size // self.num_heads
        self.num_key_value_groups = self.num_heads // self.num_kv_heads
        
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)
        
        self.rotary_emb = RotaryEmbedding(self.head_dim, config.max_position_embeddings)
    
    def forward(self, hidden_states, attention_mask=None):
        batch_size, seq_len, _ = hidden_states.size()
        
        q = self.q_proj(hidden_states).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        # Rotary embeddings
        cos, sin = self.rotary_emb(hidden_states, seq_len)
        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)
        q, k = apply_rotary_pos_emb(q, k, cos, sin)
        
        # Repeat KV for grouped attention
        k = k.repeat_interleave(self.num_key_value_groups, dim=1)
        v = v.repeat_interleave(self.num_key_value_groups, dim=1)
        
        # Attention
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask
        
        attn_weights = F.softmax(attn_weights, dim=-1, dtype=torch.float32).to(q.dtype)
        attn_output = torch.matmul(attn_weights, v)
        
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
        return self.o_proj(attn_output)

class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
    
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class TransformerBlock(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.layer_idx = layer_idx
        self.input_layernorm = RMSNorm(config.hidden_size)
        self.self_attn = GroupedQueryAttention(config)
        self.post_attention_layernorm = RMSNorm(config.hidden_size)
        self.mlp = SwiGLU(config)
    
    def forward(self, hidden_states, attention_mask=None):
        residual = hidden_states
        hidden_states = self.input_layernorm(hidden_states)
        hidden_states = self.self_attn(hidden_states, attention_mask)
        hidden_states = residual + hidden_states
        
        residual = hidden_states
        hidden_states = self.post_attention_layernorm(hidden_states)
        hidden_states = self.mlp(hidden_states)
        hidden_states = residual + hidden_states
        
        return hidden_states

class AdaptiveSLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([TransformerBlock(config, i) for i in range(config.num_layers)])
        self.norm = RMSNorm(config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        
        # Weight tying (MobileLLM innovation)
        self.lm_head.weight = self.embed_tokens.weight
        
        self.apply(self._init_weights)
        print(f"Model Parameters: {self.count_parameters():,}")
    
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def count_parameters(self):
        return sum(p.numel() for p in self.parameters())
    
    def forward(self, input_ids, attention_mask=None, labels=None):
        batch_size, seq_len = input_ids.shape
        hidden_states = self.embed_tokens(input_ids)
        
        # Causal mask
        causal_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf"), device=input_ids.device), diagonal=1)
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)
        
        for layer in self.layers:
            hidden_states = layer(hidden_states, causal_mask)
        
        hidden_states = self.norm(hidden_states)
        logits = self.lm_head(hidden_states)
        
        loss = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss = F.cross_entropy(shift_logits.view(-1, config.vocab_size), shift_labels.view(-1), ignore_index=-100)
        
        return {"loss": loss, "logits": logits}

# Create model
model = AdaptiveSLM(config)

# Enable gradient checkpointing
if config.use_gradient_checkpointing:
    model.gradient_checkpointing_enable = lambda: None  # Placeholder
    print("Gradient checkpointing enabled")

## 3. Load Data

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, Dataset

# Use Qwen tokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Vocabulary size: {len(tokenizer)}")

# Load high-quality dataset
dataset = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True)

class StreamingDataset(Dataset):
    def __init__(self, dataset, tokenizer, max_length=2048, buffer_size=10000):
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.buffer = []
        self.dataset_iter = iter(dataset)
        self._fill_buffer(buffer_size)
    
    def _fill_buffer(self, size):
        for _ in range(size):
            try:
                item = next(self.dataset_iter)
                text = item.get("text", "")
                if len(text) > 50:  # Quality filter
                    self.buffer.append(text)
            except StopIteration:
                break
    
    def __len__(self):
        return len(self.buffer)
    
    def __getitem__(self, idx):
        text = self.buffer[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": encoding["input_ids"].squeeze(0).clone()
        }

# Create dataset
train_dataset = StreamingDataset(dataset, tokenizer, buffer_size=50000)
print(f"Dataset size: {len(train_dataset)}")

train_loader = DataLoader(
    train_dataset,
    batch_size=config.batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

## 4. Knowledge Distillation Setup

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

# Load teacher model (4-bit quantized to save memory)
print("Loading teacher model (Qwen-7B)...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

teacher_model = AutoModelForCausalLM.from_pretrained(
    config.teacher_model,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)
teacher_model.eval()

print("Teacher model loaded!")

In [ ]:
class DistillationLoss(nn.Module):
    """Knowledge Distillation + PAKD Loss"""
    
    def __init__(self, alpha=0.5, temperature=2.0):
        super().__init__()
        self.alpha = alpha
        self.temperature = temperature
    
    def forward(self, student_logits, teacher_logits, labels, vocab_size):
        # Cross-entropy loss
        ce_loss = F.cross_entropy(
            student_logits.view(-1, vocab_size),
            labels.view(-1),
            ignore_index=-100
        )
        
        # KL divergence with teacher
        student_probs = F.log_softmax(student_logits / self.temperature, dim=-1)
        teacher_probs = F.softmax(teacher_logits / self.temperature, dim=-1)
        
        kd_loss = F.kl_div(
            student_probs.view(-1, vocab_size),
            teacher_probs.view(-1, vocab_size),
            reduction="batchmean"
        ) * (self.temperature ** 2)
        
        return (1 - self.alpha) * ce_loss + self.alpha * kd_loss

kd_loss_fn = DistillationLoss(alpha=config.distillation_alpha, temperature=config.temperature)

## 5. Training Loop

In [ ]:
from tqdm.auto import tqdm
import wandb

# Initialize wandb (optional)
# wandb.init(project="adaptive-slm", name="kaggle-training")

# Move model to device
model = model.to(DEVICE)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=config.learning_rate,
    weight_decay=config.weight_decay,
    betas=(0.9, 0.95)
)

# Scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=config.max_steps
)

# Mixed precision
scaler = torch.cuda.amp.GradScaler() if config.use_mixed_precision and not USE_TPU else None

# Training
model.train()
global_step = 0
running_loss = 0.0

pbar = tqdm(total=config.max_steps, desc="Training")

while global_step < config.max_steps:
    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        
        # Get teacher logits (no grad)
        with torch.no_grad():
            teacher_outputs = teacher_model(input_ids=input_ids, attention_mask=attention_mask)
            teacher_logits = teacher_outputs.logits
        
        # Forward pass with mixed precision
        with torch.cuda.amp.autocast(enabled=config.use_mixed_precision and not USE_TPU):
            student_outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            student_logits = student_outputs["logits"]
            
            # Distillation loss
            loss = kd_loss_fn(
                student_logits[:, :-1],
                teacher_logits[:, :-1],
                labels[:, 1:],
                config.vocab_size
            )
            loss = loss / config.gradient_accumulation_steps
        
        # Backward
        if scaler:
            scaler.scale(loss).backward()
        else:
            loss.backward()
        
        running_loss += loss.item()
        
        # Optimizer step
        if (global_step + 1) % config.gradient_accumulation_steps == 0:
            if scaler:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            
            optimizer.zero_grad()
            scheduler.step()
        
        global_step += 1
        pbar.update(1)
        
        # Logging
        if global_step % 100 == 0:
            avg_loss = running_loss / 100
            pbar.set_postfix({"loss": f"{avg_loss:.4f}", "lr": f"{scheduler.get_last_lr()[0]:.2e}"})
            running_loss = 0.0
        
        # Save checkpoint
        if global_step % 2000 == 0:
            os.makedirs(config.output_dir, exist_ok=True)
            torch.save({
                "step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
            }, f"{config.output_dir}/checkpoint_{global_step}.pt")
            print(f"\nSaved checkpoint at step {global_step}")
        
        if global_step >= config.max_steps:
            break

pbar.close()
print("Training complete!")

## 6. Save Final Model

In [ ]:
# Save final checkpoint
os.makedirs(config.output_dir, exist_ok=True)
torch.save({
    "model_state_dict": model.state_dict(),
    "config": config.__dict__
}, f"{config.output_dir}/adaptive_slm_final.pt")

print(f"Model saved to {config.output_dir}/adaptive_slm_final.pt")

# Also save for download
!cp {config.output_dir}/adaptive_slm_final.pt /kaggle/working/adaptive_slm_final.pt

## 7. Quick Evaluation

In [ ]:
# Quick generation test
model.eval()

test_prompt = "What is machine learning? Explain simply."
inputs = tokenizer(test_prompt, return_tensors="pt").to(DEVICE)

with torch.no_grad():
    outputs = model(input_ids=inputs["input_ids"])
    next_token_logits = outputs["logits"][0, -1]
    next_token = torch.argmax(next_token_logits)
    
print(f"Prompt: {test_prompt}")
print(f"Next token: {tokenizer.decode([next_token])}")

# Memory usage
if torch.cuda.is_available():
    print(f"\nGPU Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## Next Steps

1. **Download the checkpoint** from `/kaggle/working/adaptive_slm_final.pt`
2. **Run MMLU evaluation** locally
3. **Export to GGUF** for inference
4. **Write the paper!**